[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebnbrkr/marv/blob/main/notebooks/marv_tests_colab.ipynb)

# MARV tests on Colab

1. The unit tests (tiny synthetic Llama, no download).
2. Checks on a real model, SmolLM2-135M: exactness, target tokens, lineage,
   provenance and scale. Each prints PASS or stops with an error. CPU on purpose,
   so reduced-precision GPU maths can't blur the exactness checks.

In [ ]:
!git clone -q https://github.com/thebnbrkr/marv
%cd marv
!pip install -q -e ".[dev]"

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"

In [ ]:
import numpy as np, torch
import marv
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float32).eval()

def check(name, ok, detail=""):
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))
    assert ok, name

In [ ]:
prompt = "The capital of France is"
w = marv.capture_writes(model, tok, prompt)
check("writes rebuild the final residual", w.reconstruction_error() < 1e-4, f"{w.reconstruction_error():.1e}")
d = marv.decompose_logit(model, tok, prompt, " Paris", " Rome")
check("decompose_logit adds up", d.check_error < 1e-3, f"{d.check_error:.1e}")
check("model prefers Paris", d.actual > 0, f"logit gap {d.actual:+.2f}")

from marv.evaluate import _target_token_ids
from marv.trace import _token_id
check('" Paris" is scored as Paris, not a space', tok.decode([_token_id(tok, " Paris")]) == " Paris")
check("no prefix pieces among Paris candidates",
      all(len(tok.decode([t]).strip()) > 3 for t in _target_token_ids(tok, "Paris")))

In [ ]:
v = marv.extract(model)
check("tied lm_head stored once", v.tied)
check("provenance recorded", bool(v.revision) and v.source_dtype == "float32" and v.exact,
      f"revision {v.revision[:10]}, {v.source_dtype}")
check("same model is one lineage", marv.lineage_score(v, v) > 0.99)

null = marv.null_model(model)
try:
    marv.diff(v, marv.extract(null))
    check("diff refuses an unrelated model", False)
except ValueError:
    check("diff refuses an unrelated model", True)
sc = marv.compare_scale(model, model, tok, prompt)
check("compare_scale: a model matches itself", sc.ok and abs(sc.final_ratio - 1) < 1e-6)
print("null model vs real:", marv.compare_scale(model, null, tok, prompt))

In [ ]:
hits = marv.active_features(model, tok, v, prompt, [20, 25], k=3)
check("active_features runs", len(hits) == 6)
for h in hits[:3]:
    print(f"  L{h.layer} f{h.feature} act={h.activation:+.2f} up={tok.batch_decode([[t] for t in h.pushes_up[:3]])}")

battery = marv.capital_edit_battery("France", "Paris", neighbours=("Italy", "Spain"))
select, held_out = marv.split_probes(battery, tags=("target", "neighbour"))
check("selection and evaluation prompts are disjoint",
      not {p.prompt for p in select} & {p.prompt for p in held_out})
feats = [(h.layer, h.feature) for h in hits[:3]]
rep = marv.study_edit(model, tok, marv.suppress(model, feats), held_out[:40])
check("study_edit on held-out prompts runs", len(rep.rows) == 40)
print("\nall checks passed")